In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Define project directories
DATA_DIR = Path("../data/raw")
REPORT_DIR = Path("../reports")

REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Load datasets
train = pd.read_csv(DATA_DIR / "train.csv")
campaigns = pd.read_csv(DATA_DIR / "campaign_data.csv")
customers = pd.read_csv(DATA_DIR / "customer_demographics.csv")
coupon_items = pd.read_csv(DATA_DIR / "coupon_item_mapping.csv")
items = pd.read_csv(DATA_DIR / "item_data.csv")
transactions = pd.read_csv(DATA_DIR / "customer_transaction_data.csv")

# Display dataset dimensions
datasets = {
    "Training": train,
    "Campaigns": campaigns,
    "Customers": customers,
    "Coupon-item mapping": coupon_items,
    "Items": items,
    "Transactions": transactions
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

Training: (78369, 5)
Campaigns: (28, 4)
Customers: (760, 7)
Coupon-item mapping: (92663, 2)
Items: (74066, 4)
Transactions: (1324566, 7)


In [2]:
# Display columns used to connect datasets

print("TRAINING DATA")
print(train.columns.tolist())

print("\nCAMPAIGN DATA")
print(campaigns.columns.tolist())

print("\nCUSTOMER DEMOGRAPHICS")
print(customers.columns.tolist())

print("\nCOUPON-ITEM MAPPING")
print(coupon_items.columns.tolist())

print("\nITEM DATA")
print(items.columns.tolist())

print("\nTRANSACTION DATA")
print(transactions.columns.tolist())

TRAINING DATA
['id', 'campaign_id', 'coupon_id', 'customer_id', 'redemption_status']

CAMPAIGN DATA
['campaign_id', 'campaign_type', 'start_date', 'end_date']

CUSTOMER DEMOGRAPHICS
['customer_id', 'age_range', 'marital_status', 'rented', 'family_size', 'no_of_children', 'income_bracket']

COUPON-ITEM MAPPING
['coupon_id', 'item_id']

ITEM DATA
['item_id', 'brand', 'brand_type', 'category']

TRANSACTION DATA
['date', 'customer_id', 'item_id', 'quantity', 'selling_price', 'other_discount', 'coupon_discount']


In [3]:
# Define the keys used to connect datasets
key_checks = {
    "Training": (train, ["campaign_id", "coupon_id", "customer_id"]),
    "Campaigns": (campaigns, ["campaign_id"]),
    "Customers": (customers, ["customer_id"]),
    "Coupon-item mapping": (coupon_items, ["coupon_id", "item_id"]),
    "Items": (items, ["item_id"]),
    "Transactions": (transactions, ["customer_id", "item_id"])
}

for name, (df, keys) in key_checks.items():
    print(f"\n--- {name} ---")

    for key in keys:
        print(f"{key}:")
        print("  Missing:", df[key].isna().sum())
        print("  Unique values:", df[key].nunique())


--- Training ---
campaign_id:
  Missing: 0
  Unique values: 18
coupon_id:
  Missing: 0
  Unique values: 866
customer_id:
  Missing: 0
  Unique values: 1428

--- Campaigns ---
campaign_id:
  Missing: 0
  Unique values: 28

--- Customers ---
customer_id:
  Missing: 0
  Unique values: 760

--- Coupon-item mapping ---
coupon_id:
  Missing: 0
  Unique values: 1116
item_id:
  Missing: 0
  Unique values: 36289

--- Items ---
item_id:
  Missing: 0
  Unique values: 74066

--- Transactions ---
customer_id:
  Missing: 0
  Unique values: 1582
item_id:
  Missing: 0
  Unique values: 74063


In [4]:
# Check whether campaign IDs are unique in the campaign dataset
print(
    "Unique campaign IDs:",
    campaigns["campaign_id"].nunique()
)
print(
    "Total campaign records:",
    len(campaigns)
)

# Merge training records with campaign information
train_campaign = train.merge(
    campaigns,
    on="campaign_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

# Check merge results
print("\nMerged dataset shape:", train_campaign.shape)

print("\nCampaign matching results:")
display(train_campaign["_merge"].value_counts())

# Display sample merged records
display(train_campaign.head())

# Remove the temporary merge indicator
train_campaign.drop(columns=["_merge"], inplace=True)

Unique campaign IDs: 28
Total campaign records: 28

Merged dataset shape: (78369, 9)

Campaign matching results:


_merge
both          78369
left_only         0
right_only        0
Name: count, dtype: int64

,id,campaign_id,coupon_id,customer_id,redemption_status,campaign_type,start_date,end_date,_merge
0,1,13,27,1053,0,X,19/05/13,05/07/13,both
1,2,13,116,48,0,X,19/05/13,05/07/13,both
2,6,9,635,205,0,Y,11/03/13,12/04/13,both
3,7,13,644,1050,0,X,19/05/13,05/07/13,both
4,9,8,1017,1489,0,X,16/02/13,05/04/13,both


In [5]:
# Merge training and campaign data with customer demographics
train_campaign_customer = train_campaign.merge(
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

# Check merge results
print("Merged dataset shape:", train_campaign_customer.shape)

print("\nCustomer matching results:")
display(train_campaign_customer["_merge"].value_counts())

# Calculate customer matching percentage
matched = (train_campaign_customer["_merge"] == "both").sum()
total = len(train_campaign_customer)

print("\nMatching Statistics:")
print("Matched records:", matched)
print("Unmatched records:", total - matched)
print("Match percentage:", round(matched / total * 100, 2), "%")

# Display sample records
display(train_campaign_customer.head())

# Remove temporary merge indicator
train_campaign_customer.drop(columns=["_merge"], inplace=True)

Merged dataset shape: (78369, 15)

Customer matching results:


_merge
both          43661
left_only     34708
right_only        0
Name: count, dtype: int64


Matching Statistics:
Matched records: 43661
Unmatched records: 34708
Match percentage: 55.71 %


,id,campaign_id,coupon_id,customer_id,redemption_status,campaign_type,start_date,end_date,age_range,marital_status,rented,family_size,no_of_children,income_bracket,_merge
0,1,13,27,1053,0,X,19/05/13,05/07/13,46-55,NaN,0.0,1,NaN,5.0,both
1,2,13,116,48,0,X,19/05/13,05/07/13,36-45,Married,0.0,2,NaN,3.0,both
2,6,9,635,205,0,Y,11/03/13,12/04/13,46-55,Married,0.0,2,NaN,7.0,both
3,7,13,644,1050,0,X,19/05/13,05/07/13,NaN,NaN,NaN,NaN,NaN,NaN,left_only
4,9,8,1017,1489,0,X,16/02/13,05/04/13,46-55,Married,0.0,2,NaN,3.0,both


In [6]:
# Check whether item IDs are unique in the item dataset
print("Total item records:", len(items))
print("Unique item IDs:", items["item_id"].nunique())

# Merge coupon-item mapping with item information
coupon_item_details = coupon_items.merge(
    items,
    on="item_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

# Check matching results
print("\nCoupon-item details shape:", coupon_item_details.shape)

print("\nItem matching results:")
display(coupon_item_details["_merge"].value_counts())

# Calculate item matching percentage
matched_items = (coupon_item_details["_merge"] == "both").sum()
total_mappings = len(coupon_item_details)

print("\nMatching Statistics:")
print("Matched mappings:", matched_items)
print("Unmatched mappings:", total_mappings - matched_items)
print(
    "Match percentage:",
    round(matched_items / total_mappings * 100, 2),
    "%"
)

# Display sample records
display(coupon_item_details.head())

# Remove temporary merge indicator
coupon_item_details.drop(columns=["_merge"], inplace=True)

Total item records: 74066
Unique item IDs: 74066

Coupon-item details shape: (92663, 6)

Item matching results:


_merge
both          92663
left_only         0
right_only        0
Name: count, dtype: int64


Matching Statistics:
Matched mappings: 92663
Unmatched mappings: 0
Match percentage: 100.0 %


,coupon_id,item_id,brand,brand_type,category,_merge
0,105,37,56,Local,Grocery,both
1,107,75,56,Local,Grocery,both
2,494,76,209,Established,Grocery,both
3,522,77,278,Established,Grocery,both
4,518,77,278,Established,Grocery,both


In [7]:
# Merge transactions with customer demographics
transaction_customer = transactions.merge(
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

# Check the merged dataset
print("Original transaction shape:", transactions.shape)
print("Merged dataset shape:", transaction_customer.shape)

# Check customer matching results
print("\nCustomer matching results:")
display(transaction_customer["_merge"].value_counts())

# Calculate matching percentage
matched = (transaction_customer["_merge"] == "both").sum()
total = len(transaction_customer)

print("\nMatching Statistics:")
print("Matched transaction records:", matched)
print("Unmatched transaction records:", total - matched)
print("Match percentage:", round(matched / total * 100, 2), "%")

# Display sample records
display(transaction_customer.head())

# Remove the temporary merge indicator
transaction_customer.drop(columns=["_merge"], inplace=True)

Original transaction shape: (1324566, 7)
Merged dataset shape: (1324566, 14)

Customer matching results:


_merge
both          791602
left_only     532964
right_only         0
Name: count, dtype: int64


Matching Statistics:
Matched transaction records: 791602
Unmatched transaction records: 532964
Match percentage: 59.76 %


,date,customer_id,item_id,quantity,selling_price,other_discount,coupon_discount,age_range,marital_status,rented,family_size,no_of_children,income_bracket,_merge
0,2012-01-02,1501,26830,1,35.26,-10.69,0.0,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1,2012-01-02,1501,54253,1,53.43,-13.89,0.0,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2,2012-01-02,1501,31962,1,106.50,-14.25,0.0,NaN,NaN,NaN,NaN,NaN,NaN,left_only
3,2012-01-02,1501,33647,1,67.32,0.00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,left_only
4,2012-01-02,1501,48199,1,71.24,-28.14,0.0,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [8]:
# Validate training and campaign integration
print("TRAINING INTEGRATION")
print("Original training rows:", len(train))
print("Integrated training rows:", len(train_campaign))
print("Rows preserved:", len(train) == len(train_campaign))

# Validate customer integration
print("\nCUSTOMER INTEGRATION")
print("Original training rows:", len(train))
print("Integrated training-customer rows:", len(train_campaign_customer))
print(
    "Rows preserved:",
    len(train) == len(train_campaign_customer)
)

# Validate coupon-item integration
print("\nCOUPON-ITEM INTEGRATION")
print("Original mapping rows:", len(coupon_items))
print("Integrated mapping rows:", len(coupon_item_details))
print(
    "Rows preserved:",
    len(coupon_items) == len(coupon_item_details)
)

# Validate transaction-customer integration
print("\nTRANSACTION-CUSTOMER INTEGRATION")
print("Original transaction rows:", len(transactions))
print("Integrated transaction rows:", len(transaction_customer))
print(
    "Rows preserved:",
    len(transactions) == len(transaction_customer)
)

# Check missing values in the integrated training dataset
print("\nMissing values in training-campaign-customer dataset:")
display(
    train_campaign_customer.isnull().sum()
    .sort_values(ascending=False)
    .head(10)
)

TRAINING INTEGRATION
Original training rows: 78369
Integrated training rows: 78369
Rows preserved: True

CUSTOMER INTEGRATION
Original training rows: 78369
Integrated training-customer rows: 78369
Rows preserved: True

COUPON-ITEM INTEGRATION
Original mapping rows: 92663
Integrated mapping rows: 92663
Rows preserved: True

TRANSACTION-CUSTOMER INTEGRATION
Original transaction rows: 1324566
Integrated transaction rows: 1324566
Rows preserved: True

Missing values in training-campaign-customer dataset:


no_of_children       64313
marital_status       52975
age_range            34708
rented               34708
income_bracket       34708
family_size          34708
campaign_type            0
redemption_status        0
customer_id              0
coupon_id                0
dtype: int64

In [9]:
# Create a dataset integration summary

integration_summary = pd.DataFrame([
    {
        "integration": "Training + Campaigns",
        "original_rows": len(train),
        "integrated_rows": len(train_campaign),
        "rows_preserved": len(train) == len(train_campaign)
    },
    {
        "integration": "Training + Campaigns + Customers",
        "original_rows": len(train),
        "integrated_rows": len(train_campaign_customer),
        "rows_preserved": len(train) == len(train_campaign_customer)
    },
    {
        "integration": "Coupon Mapping + Items",
        "original_rows": len(coupon_items),
        "integrated_rows": len(coupon_item_details),
        "rows_preserved": len(coupon_items) == len(coupon_item_details)
    },
    {
        "integration": "Transactions + Customers",
        "original_rows": len(transactions),
        "integrated_rows": len(transaction_customer),
        "rows_preserved": len(transactions) == len(transaction_customer)
    }
])

display(integration_summary)

# Save the report
integration_summary.to_csv(
    REPORT_DIR / "day07_integration_summary.csv",
    index=False
)

print("\nIntegration summary saved successfully!")

,integration,original_rows,integrated_rows,rows_preserved
0,Training + Campaigns,78369,78369,True
1,Training + Campaigns + Customers,78369,78369,True
2,Coupon Mapping + Items,92663,92663,True
3,Transactions + Customers,1324566,1324566,True



Integration summary saved successfully!


In [10]:
# Inspect available variables for causal analysis

print("TRAINING DATA")
print(train.columns.tolist())

print("\nTRANSACTION DATA")
print(transactions.columns.tolist())

print("\nCAMPAIGN DATA")
print(campaigns.columns.tolist())

# Check treatment-related variables
print("\nDiscount-related variables:")
discount_columns = [
    col for col in transactions.columns
    if "discount" in col.lower()
]

print(discount_columns)

# Check redemption outcome distribution
print("\nRedemption outcome distribution:")
display(train["redemption_status"].value_counts())

# Check whether transaction records contain redemption outcomes
print(
    "\nDoes transaction data contain redemption_status?",
    "redemption_status" in transactions.columns
)

TRAINING DATA
['id', 'campaign_id', 'coupon_id', 'customer_id', 'redemption_status']

TRANSACTION DATA
['date', 'customer_id', 'item_id', 'quantity', 'selling_price', 'other_discount', 'coupon_discount']

CAMPAIGN DATA
['campaign_id', 'campaign_type', 'start_date', 'end_date']

Discount-related variables:
['other_discount', 'coupon_discount']

Redemption outcome distribution:


redemption_status
0    77640
1      729
Name: count, dtype: int64


Does transaction data contain redemption_status? False


In [11]:
# Summarize variables relevant to future causal analysis

causal_variable_summary = pd.DataFrame({
    "category": [
        "Potential outcome",
        "Potential treatment",
        "Potential treatment",
        "Potential covariate",
        "Potential covariate",
        "Potential covariate"
    ],
    "variable": [
        "redemption_status",
        "coupon_discount",
        "other_discount",
        "customer_id",
        "campaign_id",
        "item_id"
    ],
    "dataset": [
        "train.csv",
        "customer_transaction_data.csv",
        "customer_transaction_data.csv",
        "Multiple datasets",
        "train.csv / campaign_data.csv",
        "customer_transaction_data.csv / item_data.csv"
    ],
    "status": [
        "Observed redemption outcome",
        "Recorded transaction discount; treatment role unverified",
        "Recorded transaction discount; treatment role unverified",
        "Identifier; not automatically a useful model feature",
        "Identifier; campaign characteristics require inspection",
        "Identifier; item characteristics require inspection"
    ]
})

display(causal_variable_summary)

causal_variable_summary.to_csv(
    REPORT_DIR / "day07_causal_variable_summary.csv",
    index=False
)

print("\nCausal variable summary saved successfully!")

,category,variable,dataset,status
0,Potential outcome,redemption_status,train.csv,Observed redemption outcome
1,Potential treatment,coupon_discount,customer_transaction_data.csv,Recorded transaction discount; treatment role ...
2,Potential treatment,other_discount,customer_transaction_data.csv,Recorded transaction discount; treatment role ...
3,Potential covariate,customer_id,Multiple datasets,Identifier; not automatically a useful model f...
4,Potential covariate,campaign_id,train.csv / campaign_data.csv,Identifier; campaign characteristics require i...
5,Potential covariate,item_id,customer_transaction_data.csv / item_data.csv,Identifier; item characteristics require inspe...



Causal variable summary saved successfully!
